# Лабораторная работа №5
## Ансамбли моделей машинного обучения. Часть 1.

**Выполнил:** Зазовский В.Д.  
**Группа:** ИБМ3-64Б

### Цель работы
Изучение ансамблей моделей машинного обучения.

### Задание
- Две модели бэггинга (Bagging, Random Forest)
- AdaBoost
- Градиентный бустинг


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Данные
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# === 1. BAGGING ===
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=4), n_estimators=100, random_state=42)
bagging.fit(X_train, y_train)
bag_pred = bagging.predict(X_test)
bag_acc = accuracy_score(y_test, bag_pred)
bag_f1 = f1_score(y_test, bag_pred, average='weighted')
print(f"Bagging: accuracy={bag_acc:.4f}, F1={bag_f1:.4f}")

# === 2. RANDOM FOREST ===
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')
print(f"Random Forest: accuracy={rf_acc:.4f}, F1={rf_f1:.4f}")


In [ ]:
# === 3. ADABOOST ===
ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2), n_estimators=100, random_state=42)
ada.fit(X_train, y_train)
ada_pred = ada.predict(X_test)
ada_acc = accuracy_score(y_test, ada_pred)
ada_f1 = f1_score(y_test, ada_pred, average='weighted')
print(f"AdaBoost: accuracy={ada_acc:.4f}, F1={ada_f1:.4f}")

# === 4. GRADIENT BOOSTING ===
gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_acc = accuracy_score(y_test, gb_pred)
gb_f1 = f1_score(y_test, gb_pred, average='weighted')
print(f"Gradient Boosting: accuracy={gb_acc:.4f}, F1={gb_f1:.4f}")


In [ ]:
# === СРАВНЕНИЕ ===
results = pd.DataFrame({
    'Модель': ['Bagging', 'Random Forest', 'AdaBoost', 'Gradient Boosting'],
    'Accuracy': [bag_acc, rf_acc, ada_acc, gb_acc],
    'F1-score': [bag_f1, rf_f1, ada_f1, gb_f1]
}).round(4)
print(results.to_string(index=False))

# Визуализация
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results))
w = 0.35
b1 = ax.bar(x - w/2, results['Accuracy'], w, label='Accuracy', color='steelblue')
b2 = ax.bar(x + w/2, results['F1-score'], w, label='F1-score', color='coral')
ax.set_ylabel('Score')
ax.set_title('Сравнение ансамблевых моделей', fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(results['Модель'], rotation=15, ha='right')
ax.legend(); ax.set_ylim(0.85, 1.05); ax.grid(True, alpha=0.3, axis='y')
for b in b1+b2:
    h = b.get_height()
    ax.annotate(f'{h:.3f}', xy=(b.get_x()+b.get_width()/2, h), xytext=(0,3),
                textcoords="offset points", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab5/ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Важность признаков Random Forest
rf_importance = pd.DataFrame({'feature': iris.feature_names, 'importance': rf.feature_importances_}).sort_values('importance', ascending=True)
gb_importance = pd.DataFrame({'feature': iris.feature_names, 'importance': gb.feature_importances_}).sort_values('importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(rf_importance['feature'], rf_importance['importance'], color='forestgreen')
axes[0].set_title('Random Forest', fontsize=12, fontweight='bold'); axes[0].grid(True, alpha=0.3)
axes[1].barh(gb_importance['feature'], gb_importance['importance'], color='darkorange')
axes[1].set_title('Gradient Boosting', fontsize=12, fontweight='bold'); axes[1].grid(True, alpha=0.3)
fig.suptitle('Важность признаков', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab5/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


### Выводы
1. Все ансамблевые модели показали отличные результаты (accuracy >93%).
2. Random Forest и Gradient Boosting — наиболее стабильные.
3. Наиболее важный признак — petal length (длина лепестка).
4. Бэггинг уменьшает дисперсию, бустинг уменьшает смещение.
5. Ансамбли превосходят одиночные модели за счёт комбинирования.
